In [1]:
!pip install catboost

In [2]:
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold, cross_val_predict
import re
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

In [3]:
df = pd.read_excel('/content/cases_train2 (15).xlsx')
df['Оценка'] = df['Оценка'].astype('float64')
df_test_cases = pd.read_excel('/content/Копия Test1.xlsx')
df_test_cases['Оценка'] = df_test_cases['Оценка'].astype('float64')

In [4]:
tokenizer = AutoTokenizer.from_pretrained("cointegrated/rubert-tiny2")
model = AutoModel.from_pretrained("cointegrated/rubert-tiny2")

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(83828, 312, padding_idx=0)
    (position_embeddings): Embedding(2048, 312)
    (token_type_embeddings): Embedding(2, 312)
    (LayerNorm): LayerNorm((312,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-2): 3 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=312, out_features=312, bias=True)
            (key): Linear(in_features=312, out_features=312, bias=True)
            (value): Linear(in_features=312, out_features=312, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=312, out_features=312, bias=True)
            (LayerNorm): LayerNorm((312,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
   

In [6]:
def split_sentences(text):
    parts = re.split(r"(?<=[.!?;])\s+|\n+", text.strip())
    return [s.strip() for s in parts if len(s.split()) >= 3]

In [7]:
def get_cls_embeddings(text, model=model, tokenizer=tokenizer, device=device, batch_size=16, max_length=512):
    all_cls = []
    if isinstance(text, str):
        text = [text]

    if len(text) == 0:
        return np.zeros((0, model.config.hidden_size), dtype=np.float32)
    with torch.no_grad():
      for i in range(0, len(text), batch_size):
        text_batch = text[i: i+batch_size]
        enc = tokenizer(text_batch, padding=True, truncation=True, max_length=max_length, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        out = model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"], token_type_ids=enc.get("token_type_ids"))
        cls = out.last_hidden_state[:, 0, :]
        cls_norm = F.normalize(cls, dim=1).cpu().numpy()
        all_cls.append(cls_norm)
    return np.vstack(all_cls)

In [8]:
def get_solution_embedding(text):
    text = "" if text is None else str(text)
    text = re.sub(r"<br\s*/?>", " ", text)
    sentences = split_sentences(text)
    if not sentences and text.strip():
        sentences = [text.strip()]
    if not sentences:
        return np.zeros(model.config.hidden_size, dtype=np.float32)
    emb = get_cls_embeddings(sentences)
    mean_vec = emb.mean(axis=0)
    return mean_vec / np.linalg.norm(mean_vec)


In [9]:
def get_solution_embeddings(texts):
    return np.vstack([get_solution_embedding(t) for t in texts])

In [10]:
X_emb_train = get_solution_embeddings(df["Решение кейса"].astype(str).str.replace(r'<br\s*/?>', ' ', regex=True))
X_emb_test = get_solution_embeddings(df_test_cases["Решение кейса"].astype(str).str.replace(r'<br\s*/?>', ' ', regex=True))

In [11]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [12]:
# Модель для оценивания критерия Анализ ЦА
y_train_audience = df["ЦА"].astype(float).values
y_test_audience = df_test_cases["ЦА"].astype(float).values
model_audience = CatBoostRegressor(iterations=300, depth=3, learning_rate=0.05,
                                   l2_leaf_reg=10, rsm=0.5, verbose=0, random_seed=42)
pred_cv_audience = np.clip(cross_val_predict(model_audience, X_emb_train, y_train_audience, cv=kf), 1, 5)
print("MAE Оценка ЦА Val", mean_absolute_error(y_train_audience, pred_cv_audience))
model_audience.fit(X_emb_train, y_train_audience)
pred_test_audience = np.clip(model_audience.predict(X_emb_test), 1, 5)
print("MAE Оценка ЦА Test",mean_absolute_error(y_test_audience, pred_test_audience))

MAE Оценка ЦА Val 0.615840437998804
MAE Оценка ЦА Test 0.6528240657041687


In [13]:
#Модель для оценивания критерия Проработка решения
y_train_solution = df["Проработка решения"].astype(float).values
y_test_solution = df_test_cases["Проработка решения"].astype(float).values
model_solution = CatBoostRegressor(iterations=300, depth=3, learning_rate=0.05,
                                   l2_leaf_reg=10, rsm=0.5, verbose=0, random_seed=42)
pred_cv_solution = np.clip(cross_val_predict(model_solution, X_emb_train, y_train_solution, cv=kf), 1, 5)
print("MAE Val Проработка решения", mean_absolute_error(y_train_solution, pred_cv_solution))
model_solution.fit(X_emb_train, y_train_solution)
pred_test_solution = np.clip(model_solution.predict(X_emb_test), 1, 5)
print("MAE Test Проработка решения",mean_absolute_error(y_test_solution, pred_test_solution))

MAE Val Проработка решения 0.6220500533842154
MAE Test Проработка решения 0.6591839925290103


In [15]:
#Модель для оценивания критерия Финансовая модель и метрики
y_train_finance = df["Финансовая модель и метрики"].astype(float).values
y_test_finance = df_test_cases["Финансовая модель и метрики"].astype(float).values
model_finance = CatBoostRegressor(iterations=300, depth=3, learning_rate=0.05,
                                  l2_leaf_reg=10, rsm=0.5, verbose=0, random_seed=42)
pred_cv_finance = np.clip(cross_val_predict(model_finance, X_emb_train, y_train_finance, cv=kf), 1, 5)
print("MAE Val Финансовая модель", mean_absolute_error(y_train_finance, pred_cv_finance))
model_finance.fit(X_emb_train, y_train_finance)
pred_test_finance = np.clip(model_finance.predict(X_emb_test), 1, 5)
print("MAE Test Финансовая модель", mean_absolute_error(y_test_finance, pred_test_finance))

MAE Val Финансовая модель 0.5997586317841215
MAE Test Финансовая модель 0.6277317639507638


In [16]:
#Модель для оценивания критерия Анализ рисков
y_train_risks = df["Анализ рисков"].astype(float).values
y_test_risks = df_test_cases["Анализ рисков"].astype(float).values
model_risks = CatBoostRegressor(iterations=300, depth=3, learning_rate=0.05,
                                l2_leaf_reg=10, rsm=0.5, verbose=0, random_seed=42)
pred_cv_risks = np.clip(cross_val_predict(model_risks, X_emb_train, y_train_risks, cv=kf), 1, 5)
print("MAE Val Анализ рисков", mean_absolute_error(y_train_risks, pred_cv_risks))
model_risks.fit(X_emb_train, y_train_risks)
pred_test_risks = np.clip(model_risks.predict(X_emb_test), 1, 5)
print("MAE Test Анализ рисков", mean_absolute_error(y_test_risks, pred_test_risks))

MAE Val Анализ рисков 0.5946465612018812
MAE Test Анализ рисков 0.7061521170525733


In [17]:
#Модель для оценивания критерия Доказательства
y_train_proves = df["Доказательства"].astype(float).values
y_test_proves = df_test_cases["Доказательства"].astype(float).values
model_proves = CatBoostRegressor(iterations=300, depth=3, learning_rate=0.05,
                                 l2_leaf_reg=10, rsm=0.5, verbose=0, random_seed=42)
pred_cv_proves = np.clip(cross_val_predict(model_proves, X_emb_train, y_train_proves, cv=kf), 1, 5)
print("MAE Val Доказательства", mean_absolute_error(y_train_proves, pred_cv_proves))
model_proves.fit(X_emb_train, y_train_proves)
pred_test_proves = np.clip(model_proves.predict(X_emb_test), 1, 5)
print("MAE Test Доказательства", mean_absolute_error(y_test_proves, pred_test_proves))

MAE Val Доказательства 0.6032277897531351
MAE Test Доказательства 0.6593470051808348


In [18]:
# Предсказание общей оценки
pred_cv_total = (pred_cv_audience + pred_cv_solution + pred_cv_finance
                 + pred_cv_risks + pred_cv_proves) / 5
pred_test_total = (pred_test_audience + pred_test_solution + pred_test_finance
                   + pred_test_risks + pred_test_proves) / 5
print("Оценка MAE Val", mean_absolute_error(df["Оценка"].astype(float), pred_cv_total))
print("Оценка MAE Test", mean_absolute_error(df_test_cases["Оценка"].astype(float), pred_test_total))

Оценка MAE Val 0.4289286606565686
Оценка MAE Test 0.4264725524113786


In [20]:
import torch, transformers
print(torch.__version__, transformers.__version__)

2.11.0+cpu 5.22.0
